<a href="https://colab.research.google.com/github/Trinav-PT/Reinforcement_Learning_For_Graph_Isomorphism/blob/main/GraphIsomorphismModel.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Welcome to Colab!

In [1]:
# =========================
# INSTALL (Colab)
# =========================
!pip install torch-geometric -q

# =========================
# IMPORTS
# =========================
import networkx as nx
import random
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from tqdm import tqdm
from torch_geometric.datasets import TUDataset

# =========================
# SEED
# =========================
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)

# =========================
# LOAD DATASET
# =========================
dataset = TUDataset(root='/tmp/MUTAG', name='MUTAG')

# =========================
# UTILS
# =========================

def pyg_to_nx(data):
    G = nx.Graph()
    edge_index = data.edge_index.numpy()
    for i in range(edge_index.shape[1]):
        u, v = int(edge_index[0, i]), int(edge_index[1, i])
        G.add_edge(u, v)
    return nx.convert_node_labels_to_integers(G)

# =========================
# WL FEATURES (STRONGER)
# =========================

def wl_colors(G, iters=3):
    colors = {n: G.degree[n] for n in G.nodes()}
    for _ in range(iters):
        new_colors = {}
        for node in G.nodes():
            neigh = sorted(colors[n] for n in G.neighbors(node))
            new_colors[node] = hash((colors[node], tuple(neigh)))
        mapping = {c: i for i, c in enumerate(set(new_colors.values()))}
        colors = {n: mapping[c] for n, c in new_colors.items()}
    return colors

# =========================
# HARD NON-ISO GENERATOR
# =========================

def hard_non_iso(G):
    G2 = G.copy()
    edges = list(G2.edges())

    for _ in range(max(3, len(edges)//10)):
        if len(edges) < 2:
            break

        (u1, v1), (u2, v2) = random.sample(edges, 2)

        if len({u1, v1, u2, v2}) < 4:
            continue
        if G2.has_edge(u1, v2) or G2.has_edge(u2, v1):
            continue

        if G2.has_edge(u1, v1):
            G2.remove_edge(u1, v1)
        if G2.has_edge(u2, v2):
            G2.remove_edge(u2, v2)

        G2.add_edge(u1, v2)
        G2.add_edge(u2, v1)

        edges = list(G2.edges())

    return G2

# =========================
# DATA GENERATION
# =========================

def generate_pair(n_range=(20, 60)):
    n = random.randint(*n_range)
    p = max(0.1, min(0.3, 5.0 / n))
    G1 = nx.fast_gnp_random_graph(n, p)

    if random.random() < 0.5:
        perm = list(range(n))
        random.shuffle(perm)
        G2 = nx.relabel_nodes(G1, dict(enumerate(perm)))
        return G1, G2, True
    else:
        return G1, hard_non_iso(G1), False


def generate_real_pair():
    G1 = pyg_to_nx(random.choice(dataset))
    n = G1.number_of_nodes()

    if n < 5:
        return generate_real_pair()

    if random.random() < 0.5:
        perm = list(range(n))
        random.shuffle(perm)
        G2 = nx.relabel_nodes(G1, dict(enumerate(perm)))
        return G1, G2, True
    else:
        return G1, hard_non_iso(G1), False

# =========================
# ENVIRONMENT
# =========================

class Env:
    def reset(self, G1, G2, label):
        assert G1.number_of_nodes() == G2.number_of_nodes()

        self.G1, self.G2 = G1, G2
        self.label = label
        self.n = G1.number_of_nodes()

        self.mapping = {}
        self.used_v = set()

        self.tri1 = nx.triangles(G1)
        self.tri2 = nx.triangles(G2)
        self.clust1 = nx.clustering(G1)
        self.clust2 = nx.clustering(G2)

        self.wl1 = wl_colors(G1)
        self.wl2 = wl_colors(G2)

        return self

    def select_node(self):
        remaining = [x for x in range(self.n) if x not in self.mapping]
        return max(remaining, key=lambda x: self.G1.degree[x])

    def valid_actions(self, u):
        u_deg = self.G1.degree[u]
        u_tri = self.tri1[u]

        return [
            v for v in range(self.n)
            if v not in self.used_v
            and abs(self.G2.degree[v] - u_deg) <= 1
            and abs(self.tri2[v] - u_tri) <= 2
        ]

    def step(self, u, v):
        if not self.check_valid(u, v):
            return -8.0, True

        self.mapping[u] = v
        self.used_v.add(v)

        if len(self.mapping) == self.n:
            return (+25.0 if self.label else -25.0), True

        return +1.0, False

    def check_valid(self, u, v):
        u_nbrs = set(self.G1.neighbors(u))
        for u_old, v_old in self.mapping.items():
            if (u_old in u_nbrs) != self.G2.has_edge(v, v_old):
                return False
        return True

# =========================
# FEATURE EXTRACTION
# =========================

def extract_features(env, u, v_list):
    feats = []
    n = env.n

    u_deg = env.G1.degree[u]
    u_tri = env.tri1[u]
    u_cl = env.clust1[u]
    u_wl = env.wl1[u]
    u_nbrs = list(env.G1.neighbors(u))

    u_2hop = sum(env.G1.degree[n] for n in u_nbrs)

    max_wl = max(env.wl1.values()) + 1

    for v in v_list:
        v_deg = env.G2.degree[v]
        v_tri = env.tri2[v]
        v_cl = env.clust2[v]
        v_wl = env.wl2[v]

        v_2hop = sum(env.G2.degree[n] for n in env.G2.neighbors(v))

        consistency = 1.0
        if env.mapping:
            matches = sum(
                (u_o in u_nbrs) == env.G2.has_edge(v, v_o)
                for u_o, v_o in env.mapping.items()
            )
            consistency = matches / len(env.mapping)

        feats.append([
            abs(u_deg - v_deg) / n,
            abs(u_tri - v_tri) / n,
            abs(u_cl - v_cl),
            consistency,
            len(env.mapping) / n,
            u_deg / n,
            v_deg / n,
            abs(u_wl - v_wl) / max_wl,
            abs(u_2hop - v_2hop) / (n*n)
        ])

    return torch.tensor(feats, dtype=torch.float32)

# =========================
# MODEL
# =========================

class ActorCritic(nn.Module):
    def __init__(self):
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(9, 256),
            nn.ReLU(),
            nn.Linear(256, 256),
            nn.ReLU()
        )

        self.actor = nn.Linear(256, 1)
        self.critic = nn.Linear(256, 1)

    def forward(self, x):
        h = self.net(x)
        return self.actor(h), self.critic(h)

# =========================
# TRAINING
# =========================

model = ActorCritic()
optimizer = optim.Adam(model.parameters(), lr=1e-4)
GAMMA = 0.97

print("Training FINAL MODEL...\n")

for ep in tqdm(range(5000)):

    if random.random() < 0.7:
        G1, G2, label = generate_pair()
    else:
        G1, G2, label = generate_real_pair()

    env = Env().reset(G1, G2, label)

    log_probs, values, rewards = [], [], []
    done = False

    while not done:

        u = env.select_node()
        actions = env.valid_actions(u)

        if not actions:
            rewards.append(-5.0)
            break

        feats = extract_features(env, u, actions)

        logits, val = model(feats)
        probs = torch.softmax(logits.view(-1), dim=0)

        dist = torch.distributions.Categorical(probs)
        idx = dist.sample()

        log_probs.append(dist.log_prob(idx))
        values.append(val.view(-1)[idx])

        v = actions[idx]
        reward, done = env.step(u, v)
        rewards.append(reward)

    if not values:
        continue

    returns, G = [], 0
    for r in reversed(rewards):
        G = r + GAMMA * G
        returns.insert(0, G)

    T = min(len(returns), len(values))
    returns = torch.tensor(returns[:T])
    values = torch.stack(values[:T])
    log_probs = torch.stack(log_probs[:T])

    adv = returns - values.detach()

    loss = -(log_probs * adv).mean() + 0.5 * adv.pow(2).mean()

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

# =========================
# EVALUATION
# =========================

def evaluate(generator, trials=50):
    correct = 0

    for _ in tqdm(range(trials)):
        G1, G2, label = generator()
        env = Env().reset(G1, G2, label)

        success = True
        done = False

        while not done:
            u = env.select_node()
            actions = env.valid_actions(u)

            if not actions:
                success = False
                break

            feats = extract_features(env, u, actions)

            with torch.no_grad():
                logits, _ = model(feats)
                idx = torch.argmax(logits.view(-1)).item()

            v = actions[idx]
            reward, done = env.step(u, v)

            if reward < 0:
                success = False

        if success == label:
            correct += 1

    print(f"Accuracy: {correct/trials:.2%}")

print("\nSynthetic:")
evaluate(generate_pair)

print("\nMUTAG:")
evaluate(generate_real_pair)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 46.8 MB/s eta 0:00:00


Processing...
Done!


Training FINAL MODEL...



100%|██████████| 5000/5000 [01:17<00:00, 64.48it/s]



Synthetic:


100%|██████████| 50/50 [00:00<00:00, 127.01it/s]


Accuracy: 100.00%

MUTAG:


100%|██████████| 50/50 [00:00<00:00, 259.84it/s]

Accuracy: 84.00%


In [2]:
# =========================
# LOAD MULTIPLE DATASETS
# =========================
from torch_geometric.datasets import TUDataset

datasets = {
    "MUTAG": TUDataset(root='/tmp/MUTAG', name='MUTAG'),
    "PROTEINS": TUDataset(root='/tmp/PROTEINS', name='PROTEINS'),
    "IMDB-BINARY": TUDataset(root='/tmp/IMDB', name='IMDB-BINARY')
}

# =========================
# GENERATE REAL PAIR (DATASET-SPECIFIC)
# =========================
def generate_real_pair_ds(dataset):
    G1 = pyg_to_nx(random.choice(dataset))
    n = G1.number_of_nodes()

    if n < 5:
        return generate_real_pair_ds(dataset)

    if random.random() < 0.5:
        # isomorphic
        perm = list(range(n))
        random.shuffle(perm)
        G2 = nx.relabel_nodes(G1, dict(enumerate(perm)))
        return G1, G2, True
    else:
        # hard non-isomorphic
        return G1, hard_non_iso(G1), False

# =========================
# EVALUATION FUNCTION
# =========================
def evaluate_dataset(name, dataset, trials=50):
    correct = 0

    for _ in tqdm(range(trials)):
        G1, G2, label = generate_real_pair_ds(dataset)
        env = Env().reset(G1, G2, label)

        success = True
        done = False

        while not done:
            u = env.select_node()
            actions = env.valid_actions(u)

            if not actions:
                success = False
                break

            feats = extract_features(env, u, actions)

            with torch.no_grad():
                logits, _ = model(feats)
                idx = torch.argmax(logits.view(-1)).item()

            v = actions[idx]
            reward, done = env.step(u, v)

            if reward < 0:
                success = False

        if success == label:
            correct += 1

    print(f"{name} → Accuracy: {correct/trials:.2%}")

# =========================
# RUN BENCHMARK
# =========================

print("\n=== MULTI-DATASET BENCHMARK ===\n")

# Synthetic baseline
print("Synthetic:")
evaluate(generate_pair)

# Real datasets
for name, ds in datasets.items():
    print(f"\n{name}:")
    evaluate_dataset(name, ds)

Processing...
Done!
Processing...
Done!



=== MULTI-DATASET BENCHMARK ===

Synthetic:


100%|██████████| 50/50 [00:00<00:00, 102.30it/s]


Accuracy: 100.00%

MUTAG:


100%|██████████| 50/50 [00:00<00:00, 216.10it/s]


MUTAG → Accuracy: 80.00%

PROTEINS:


100%|██████████| 50/50 [00:01<00:00, 34.50it/s]


PROTEINS → Accuracy: 88.00%

IMDB-BINARY:


100%|██████████| 50/50 [00:00<00:00, 116.23it/s]

IMDB-BINARY → Accuracy: 100.00%


In [ ]:
# =========================
# IMPORTS
# =========================
import time
import networkx as nx
import random
from tqdm import tqdm
import torch

# =========================
# HARD GRAPH GENERATORS
# =========================
def generate_regular_graph(n, k=4):
    if (n * k) % 2 != 0:
        k -= 1
    return nx.random_regular_graph(k, n)

def generate_grid_graph(n):
    side = int(n**0.5)
    G = nx.grid_2d_graph(side, side)
    return nx.convert_node_labels_to_integers(G)

def generate_cycle_graph(n):
    return nx.cycle_graph(n)

def hard_graph_pair(n=50):
    choice = random.choice(["regular", "grid", "cycle"])

    if choice == "regular":
        G1 = generate_regular_graph(n, k=4)
    elif choice == "grid":
        G1 = generate_grid_graph(n)
    else:
        G1 = generate_cycle_graph(n)

    if random.random() < 0.5:
        perm = list(range(len(G1)))
        random.shuffle(perm)
        G2 = nx.relabel_nodes(G1, dict(enumerate(perm)))
        return G1, G2, True
    else:
        G2 = G1.copy()
        edges = list(G2.edges())

        if len(edges) >= 2:
            (u1, v1), (u2, v2) = random.sample(edges, 2)
            if len({u1, v1, u2, v2}) == 4:
                if G2.has_edge(u1, v1):
                    G2.remove_edge(u1, v1)
                if G2.has_edge(u2, v2):
                    G2.remove_edge(u2, v2)

                G2.add_edge(u1, v2)
                G2.add_edge(u2, v1)

        return G1, G2, False

# =========================
# RL vs VF2 EVALUATION
# =========================
def evaluate_hard_with_vf2(n, trials=30):

    rl_correct = 0
    vf2_correct = 0

    rl_time = 0
    vf2_time = 0

    for _ in tqdm(range(trials)):

        G1, G2, label = hard_graph_pair(n)

        # ================= RL =================
        start = time.time()

        env = Env().reset(G1, G2, label)
        done = False
        success = True

        while not done:
            u = env.select_node()
            actions = env.valid_actions(u)

            if not actions:
                success = False
                break

            feats = extract_features(env, u, actions)

            with torch.no_grad():
                logits, _ = model(feats)
                idx = torch.argmax(logits.view(-1)).item()

            v = actions[idx]
            reward, done = env.step(u, v)

            if reward < 0:
                success = False

        rl_pred = success
        rl_time += (time.time() - start)

        if rl_pred == label:
            rl_correct += 1

        # ================= VF2 =================
        start = time.time()

        matcher = nx.algorithms.isomorphism.GraphMatcher(G1, G2)
        vf2_pred = matcher.is_isomorphic()

        vf2_time += (time.time() - start)

        if vf2_pred == label:
            vf2_correct += 1

    # ================= RESULTS =================
    print(f"\n=== HARD GRAPH RESULTS (n={n}) ===")

    print(f"RL Accuracy:  {rl_correct/trials:.2%}")
    print(f"VF2 Accuracy: {vf2_correct/trials:.2%}")

    print(f"\nRL Avg Time:  {rl_time/trials:.6f} sec")
    print(f"VF2 Avg Time: {vf2_time/trials:.6f} sec")

# =========================
# RUN
# =========================

print("\n=== RL vs VF2 (HARD GRAPHS) ===\n")

evaluate_hard_with_vf2(50)
evaluate_hard_with_vf2(100)
evaluate_hard_with_vf2(200)


=== RL vs VF2 (HARD GRAPHS) ===



100%|██████████| 30/30 [00:03<00:00,  8.66it/s]



=== HARD GRAPH RESULTS (n=50) ===
RL Accuracy:  80.00%
VF2 Accuracy: 100.00%

RL Avg Time:  0.029509 sec
VF2 Avg Time: 0.084125 sec


100%|██████████| 30/30 [01:33<00:00,  3.13s/it]



=== HARD GRAPH RESULTS (n=100) ===
RL Accuracy:  80.00%
VF2 Accuracy: 100.00%

RL Avg Time:  0.067433 sec
VF2 Avg Time: 3.056066 sec


  0%|          | 0/30 [00:00<?, ?it/s]